## Problema dos Padrões

In [1]:
%pip install pulp -q

import pulp

prob = pulp.LpProblem("Padroes", pulp.LpMaximize)

# ENUNCIADO
# Uma fabrica de latinhas possui 4 padroes de impressao em folhas de metal
# (existem 2 tipos de folhas de metal diferentes).
# A fabrica possui 200 folhas de metal de tam 1 e 90 de tam 2.
# Cada latinha e vendida a 50 u.
# Cada corpo nao utilizado possui um custo de estocagem de 5 u e cada tampa custa 3 u.
# O tempo total de impressao nao pode passar de 100 s.
# Quantas impressoes de cada padrao devem ser feitas para maximizar o lucro?

I = [1, 2, 3, 4]
tam_folha = {1: 1, 2: 2, 3: 1, 4: 1}
corpo = {1: 1, 2: 2, 3: 0, 4: 4}
tampa = {1: 7, 2: 3, 3: 9, 4: 4}
tempo = {1: 2, 2: 3, 3: 2, 4: 1}
folhas_tam1 = 200
folhas_tam2 = 90
tempo_max = 100

# VARIAVEIS DE DECISAO
# x_i = qt de folhas do padrao i impressas, para todo i pertencente a {1, 2, 3, 4}
# y = qt de latinhas produzidas
x = {i: pulp.LpVariable(f"x_{i}", lowBound=0, cat='Integer') for i in I}
y = pulp.LpVariable("y", lowBound=0, cat='Integer')

# Quantidades totais de corpos e tampas produzidos
num_corpos = pulp.lpSum(corpo[i] * x[i] for i in I)
num_tampas = pulp.lpSum(tampa[i] * x[i] for i in I)

## RESTRICOES
# (dominio) x_i pertence a Z+ e y pertence a Z+ — garantido por cat='Integer' e lowBound=0

# (folhas tamanho 1) x1 + x3 + x4 <= 200
prob += x[1] + x[3] + x[4] <= folhas_tam1

# (folhas tamanho 2) x2 <= 90
prob += x[2] <= folhas_tam2

# (tempo) 2*x1 + 3*x2 + 2*x3 + x4 <= 100
prob += pulp.lpSum(tempo[i] * x[i] for i in I) <= tempo_max

# (corpo, tampa) y <= #c e y <= #t/2
prob += y <= num_corpos
prob += 2 * y <= num_tampas

# FUNCAO OBJETIVO: max 50*y - 5*(#c - y) - 3*(#t - 2*y)
prob += 50 * y - 5 * (num_corpos - y) - 3 * (num_tampas - 2 * y)

prob.solve(pulp.PULP_CBC_CMD(msg=False))

# Resultados
qtd_padroes = {i: x[i].varValue for i in I}
latinhas = y.varValue
total_corpos = pulp.value(num_corpos)
total_tampas = pulp.value(num_tampas)
tempo_usado = sum(tempo[i] * qtd_padroes[i] for i in I)
lucro = pulp.value(prob.objective)
folhas_tam1_usadas = qtd_padroes[1] + qtd_padroes[3] + qtd_padroes[4]

print("\nPROBLEMA DOS PADROES - LUCRO MAXIMO\n")
print(f"Lucro Total Maximo: {lucro:.2f} u\n")
print(f"Latinhas produzidas: {latinhas:.0f}")
print(f"Corpos produzidos: {total_corpos:.0f} (sobra: {total_corpos - latinhas:.0f})")
print(f"Tampas produzidas: {total_tampas:.0f} (sobra: {total_tampas - 2 * latinhas:.0f})\n")

print("Impressoes por padrao:")
print("Padrao\tFolhas\tCorpos/folha\tTampas/folha\tTempo/folha")
for i in I:
  print(f"{i}\t{qtd_padroes[i]:.0f}\t{corpo[i]}\t\t{tampa[i]}\t\t{tempo[i]}")

print(f"\nFolhas tam 1 utilizadas: {folhas_tam1_usadas:.0f} / {folhas_tam1}")
print(f"Folhas tam 2 utilizadas: {qtd_padroes[2]:.0f} / {folhas_tam2}")
print(f"Tempo de impressao: {tempo_usado:.0f} / {tempo_max} s")


Note: you may need to restart the kernel to use updated packages.

PROBLEMA DOS PADROES - LUCRO MAXIMO

Lucro Total Maximo: 10522.00 u

Latinhas produzidas: 211
Corpos produzidos: 216 (sobra: 5)
Tampas produzidas: 423 (sobra: 1)

Impressoes por padrao:
Padrao	Folhas	Corpos/folha	Tampas/folha	Tempo/folha
1	0	1		7		2
2	0	2		3		3
3	23	0		9		2
4	54	4		4		1

Folhas tam 1 utilizadas: 77 / 200
Folhas tam 2 utilizadas: 0 / 90
Tempo de impressao: 100 / 100 s
